In [ ]:
import time
import torch
from torchvision import datasets
from torch import nn, optim, autograd
import numpy as np
import matplotlib.pyplot as plt

# 设定随机种子
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 当前使用设备: {device} | 模式: 官方基线对齐 + LoRA双路线消融")

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#4a433a',
    'axes.labelcolor': '#2f2a24',
    'xtick.color': '#2f2a24',
    'ytick.color': '#2f2a24',
    'grid.color': '#d8d1c4',
    'grid.alpha': 0.35,


In [ ]:
    'axes.titleweight': 'bold'
})

# ==========================================
# 1. 数据构造 (保持官方 CMNIST 逻辑)
# ==========================================
def make_environment(images, labels, e):
    def torch_bernoulli(p, size): return (torch.rand(size) < p).float()
    def torch_xor(a, b): return (a-b).abs()
    images = images.reshape((-1, 28, 28))[:, ::2, ::2]
    labels = (labels < 5).float()
    labels = torch_xor(labels, torch_bernoulli(0.25, len(labels)))
    colors = torch_xor(labels, torch_bernoulli(e, len(labels)))
    images = torch.stack([images, images], dim=1)
    images[torch.tensor(range(len(images))), (1-colors).long(), :, :] *= 0
    return {'images': (images.float() / 255.).reshape(-1, 2*14*14).to(device), 
            'labels': labels[:, None].to(device)}

mnist = datasets.MNIST('./data', train=True, download=True)
envs = [
    make_environment(mnist.data[:25000], mnist.targets[:25000], 0.2),  
    make_environment(mnist.data[25000:50000], mnist.targets[25000:50000], 0.1), 
    make_environment(mnist.data[50000:], mnist.targets[50000:], 0.9)   
]
ENV_NOISE = [0.2, 0.1, 0.9]
METHOD_COLORS = {
    'IRMv1': '#4c78a8',
    'Full-BIRM (Official)': '#f58518',
    'LoRA-BIRM (Route A)': '#54a24b',
    'LoRA-BIRM (Route B)': '#e45756'
}
METHOD_DISPLAY_NAMES = {
    'IRMv1': 'IRMv1',
    'Full-BIRM (Official)': 'Full-BIRM\n(Official)',
    'LoRA-BIRM (Route A)': 'LoRA-BIRM\n(Route A)',
    'LoRA-BIRM (Route B)': 'LoRA-BIRM\n(Route B)'
}

def flat_to_rgb_image(flat_image):
    image = flat_image.detach().cpu().reshape(2, 14, 14).numpy()
    rgb = np.zeros((14, 14, 3), dtype=np.float32)
    rgb[..., 0] = image[0]
    rgb[..., 1] = image[1]
    return np.clip(rgb, 0.0, 1.0)

def infer_color_name(image):
    return 'Green' if image[..., 1].sum() >= image[..., 0].sum() else 'Red'

def plot_environment_overview(envs, noise_levels, samples_per_env=6):
    fig, axes = plt.subplots(
        len(envs),
        samples_per_env,
        figsize=(2.35 * samples_per_env, 2.55 * len(envs)),
        facecolor='#f7f3eb'
    )
    axes = np.atleast_2d(axes)
    fig.suptitle('CMNIST Environment Shift Overview', fontsize=18, fontweight='bold', x=0.02, ha='left', y=1.02)
    fig.text(0.02, 0.965, 'Train environments preserve the color shortcut; the test environment flips it aggressively.', fontsize=11, color='#5a544a')

    for row, (env, noise) in enumerate(zip(envs, noise_levels)):
        sample_ids = np.linspace(0, len(env['labels']) - 1, samples_per_env, dtype=int)
        match_scores = []

        for col, idx in enumerate(sample_ids):
            ax = axes[row, col]
            image = flat_to_rgb_image(env['images'][idx])
            label_value = int(env['labels'][idx].item())
            color_name = infer_color_name(image)
            match_scores.append(int((color_name == 'Green') == bool(label_value)))

            ax.imshow(image)
            ax.set_title(f'y={label_value} | {color_name}', fontsize=9.5, pad=6)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(1.15)
                spine.set_edgecolor('#d9d1c3')

        shortcut_match = np.mean(match_scores)
        axes[row, 0].set_ylabel(
            f'Env {row + 1}\nflip={noise:.1f}\nshortcut={shortcut_match:.2f}',
            rotation=0,
            labelpad=42,
            va='center',
            fontsize=10.5,
            fontweight='bold'
        )



In [ ]:
    plt.tight_layout()
    fig.savefig('cmnist_environment_overview.png', dpi=220, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

plot_environment_overview(envs, ENV_NOISE)

# Route A 需要的拼接全局数据
train_x = torch.cat([envs[0]['images'], envs[1]['images']])
train_y = torch.cat([envs[0]['labels'], envs[1]['labels']])
train_g = torch.cat([torch.zeros_like(envs[0]['labels']), torch.ones_like(envs[1]['labels'])]).view(-1)

# ==========================================
# 2. 核心网络组件与 EBD
# ==========================================
class EBD(nn.Module):
    def __init__(self, envs_num=2):
        super().__init__()
        self.emb = nn.Embedding(envs_num, 1)
        self.emb.weight.data.fill_(1.0) 
    def forward(self, g, noise_sd=0.0):
        w = self.emb(g.long())
        if noise_sd > 0:
            w = w + (torch.randn_like(self.emb.weight) * noise_sd)[g.long()]
        return w

# 纯确定性 MLP (供 IRMv1, Full-BIRM 官方基线使用)
class StandardMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1, self.lin2, self.lin3 = nn.Linear(392, 256), nn.Linear(256, 256), nn.Linear(256, 1)
        self.relu = nn.ReLU()
    def forward(self, x, sample=False): # sample 占位符
        return self.lin3(self.relu(self.lin2(self.relu(self.lin1(x)))))

# 确定性 LoRA (供 LoRA-BIRM 路线A使用)
class DetLoRALinear(nn.Module):
    def __init__(self, in_f, out_f, r=8):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
        self.A = nn.Parameter(torch.randn(r, in_f) / np.sqrt(in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
    def forward(self, x):
        return self.linear(x) + (x @ self.A.t() @ self.B.t()) * (8 / self.A.size(0))

class DetLoRAMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1, self.lin2, self.lin3 = DetLoRALinear(392, 256), DetLoRALinear(256, 256), nn.Linear(256, 1)
        self.relu = nn.ReLU()
    def forward(self, x, sample=False):
        return self.lin3(self.relu(self.lin2(self.relu(self.lin1(x)))))

# 真实的变分 LoRA (供 LoRA-BIRM 路线B使用)
class TrueBayesLoRALinear(nn.Module):
    def __init__(self, in_f, out_f, r=8):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
        self.A = nn.Parameter(torch.randn(r, in_f) / np.sqrt(in_f))
        self.B_mu = nn.Parameter(torch.zeros(out_f, r))
        self.B_logvar = nn.Parameter(torch.full((out_f, r), -5.0))
    def forward(self, x, sample=True):
        B_sample = self.B_mu + (torch.randn_like(self.B_mu) * torch.exp(0.5 * self.B_logvar) if sample else 0)
        return self.linear(x) + (x @ self.A.t() @ B_sample.t()) * (8 / self.A.size(0))

class TrueBayesLoRAMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1, self.lin2, self.lin3 = TrueBayesLoRALinear(392, 256), TrueBayesLoRALinear(256, 256), nn.Linear(256, 1)
        self.relu = nn.ReLU()
    def forward(self, x, sample=True):
        return self.lin3(self.relu(self.lin2(self.relu(self.lin1(x, sample)), sample)))

# ==========================================
# 3. 训练核心逻辑分配器
# ==========================================
def run_experiment(method_name, steps=2000):
    if method_name in ['IRMv1', 'Full-BIRM (Official)']: model = StandardMLP().to(device)
    elif method_name == 'LoRA-BIRM (Route A)': model = DetLoRAMLP().to(device)
    elif method_name == 'LoRA-BIRM (Route B)': model = TrueBayesLoRAMLP().to(device)

    is_route_A = (method_name != 'LoRA-BIRM (Route B)')
    


In [ ]:
    if is_route_A:
        ebd = EBD(envs_num=2).to(device)
        optimizer = optim.Adam(list(model.parameters()) + list(ebd.parameters()), lr=1e-3)
    else:
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_test_acc = 0.0
    start_time = time.time()

    for step in range(steps + 1):
        model.train()
        
        # --- Route A (官方 EBD 逻辑) ---
        if is_route_A:
            ebd.emb.weight.data.fill_(1.0)
            logits = model(train_x) # 前向传播仅 1 次
            
            if method_name == 'IRMv1':
                logits_w = logits * ebd(train_g, noise_sd=0.0)
                train_nll = nn.functional.binary_cross_entropy_with_logits(logits_w, train_y)
                grad = autograd.grad(train_nll * 2, ebd.parameters(), create_graph=True)[0]
                train_penalty = torch.mean(grad ** 2)
            else:
                noise_sd = 0.024 # 1200 / 50000 官方系数
                train_nll = nn.functional.binary_cross_entropy_with_logits(logits, train_y)
                train_penalty = 0.0
                for _ in range(10): # 标量上 10 次 MC
                    logits_w = logits * ebd(train_g, noise_sd=noise_sd)
                    nll_w = nn.functional.binary_cross_entropy_with_logits(logits_w, train_y)
                    grad = autograd.grad(nll_w * 2, ebd.parameters(), create_graph=True)[0]
                    train_penalty += torch.mean(grad ** 2) / 10.0
                    
        # --- Route B (真实隐空间变分推断逻辑) ---
        else:
            env_losses = []
            for env in envs[:2]:
                mc_losses = []
                for _ in range(3): # 真实矩阵上 3 次 MC
                    logits = model(env['images'], sample=True)
                    mc_losses.append(nn.functional.binary_cross_entropy_with_logits(logits, env['labels']))
                env_losses.append(torch.stack(mc_losses).mean())
            train_nll = torch.stack(env_losses).mean()
            train_penalty = torch.stack(env_losses).var(unbiased=False) # 一阶 V-REx 方差惩罚

        # --- 统一的硬退火与正则化 ---
        penalty_weight = 10000.0 if step >= 200 else 0.0
        weight_norm = sum(w.norm().pow(2) for w in model.parameters())
        
        loss = train_nll + 0.001 * weight_norm + penalty_weight * train_penalty
        if penalty_weight > 1.0: loss /= (1. + penalty_weight)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                test_logits = model(envs[2]['images'], sample=False)
                test_acc = ((test_logits > 0.).float() == envs[2]['labels']).float().mean().item()
                if test_acc > best_test_acc: best_test_acc = test_acc

    return best_test_acc * 100.0, time.time() - start_time

def build_summary(results):
    summary = {}
    for method, metric in results.items():
        acc = np.asarray(metric['acc'], dtype=float)
        runtime = np.asarray(metric['time'], dtype=float)
        summary[method] = {
            'acc_mean': acc.mean(),
            'acc_std': acc.std(),
            'time_mean': runtime.mean(),
            'time_std': runtime.std()
        }
    return summary


In [ ]:

def style_axis(ax, grid_axis='both'):
    ax.grid(True, axis=grid_axis, linestyle='--', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_facecolor('#fffdf8')

def plot_results_dashboard(results):
    summary = build_summary(results)
    method_order = list(results.keys())
    labels = [METHOD_DISPLAY_NAMES[m] for m in method_order]
    colors = [METHOD_COLORS[m] for m in method_order]
    acc_runs = [np.asarray(results[m]['acc'], dtype=float) for m in method_order]
    acc_means = np.asarray([summary[m]['acc_mean'] for m in method_order])
    acc_stds = np.asarray([summary[m]['acc_std'] for m in method_order])
    time_means = np.asarray([summary[m]['time_mean'] for m in method_order])


In [ ]:
    time_stds = np.asarray([summary[m]['time_std'] for m in method_order])
    y_pos = np.arange(len(method_order))

    fig = plt.figure(figsize=(16, 10), constrained_layout=True, facecolor='#f7f3eb')
    grid = fig.add_gridspec(2, 2, hspace=0.12, wspace=0.10)
    ax_acc = fig.add_subplot(grid[0, 0])
    ax_time = fig.add_subplot(grid[0, 1])
    ax_tradeoff = fig.add_subplot(grid[1, 0])
    ax_runs = fig.add_subplot(grid[1, 1])

    fig.suptitle('CMNIST Causal Generalization Dashboard', fontsize=18, fontweight='bold', x=0.02, ha='left')
    best_method = method_order[int(np.argmax(acc_means))]
    fastest_method = method_order[int(np.argmin(time_means))]
    fig.text(0.02, 0.955, f'Best accuracy: {best_method} | Fastest runtime: {fastest_method}', fontsize=11, color='#5a544a')

    style_axis(ax_acc, grid_axis='x')
    ax_acc.barh(y_pos, acc_means, xerr=acc_stds, color=colors, alpha=0.92, edgecolor='none', error_kw={'ecolor': '#2f2a24', 'capsize': 4, 'elinewidth': 1.1})
    ax_acc.set_yticks(y_pos, labels)
    ax_acc.invert_yaxis()
    ax_acc.set_xlabel('Best OOD accuracy (%)')
    ax_acc.set_title('Accuracy summary', loc='left', fontsize=14)
    for idx, (mean, std) in enumerate(zip(acc_means, acc_stds)):
        ax_acc.text(mean + 0.4, idx, f'{mean:.2f}±{std:.2f}', va='center', fontsize=10, color='#2f2a24')

    style_axis(ax_time, grid_axis='x')
    ax_time.barh(y_pos, time_means, xerr=time_stds, color=colors, alpha=0.92, edgecolor='none', error_kw={'ecolor': '#2f2a24', 'capsize': 4, 'elinewidth': 1.1})
    ax_time.set_yticks(y_pos, labels)
    ax_time.invert_yaxis()
    ax_time.set_xlabel('Runtime per run (s)')
    ax_time.set_title('Efficiency summary', loc='left', fontsize=14)
    for idx, (mean, std) in enumerate(zip(time_means, time_stds)):
        ax_time.text(mean + 0.4, idx, f'{mean:.1f}±{std:.1f}s', va='center', fontsize=10, color='#2f2a24')

    style_axis(ax_tradeoff)
    ax_tradeoff.scatter(time_means, acc_means, s=230, c=colors, edgecolor='#1f1b16', linewidth=1.1)
    ax_tradeoff.set_xlabel('Mean runtime per run (s)')
    ax_tradeoff.set_ylabel('Mean best OOD accuracy (%)')
    ax_tradeoff.set_title('Accuracy vs runtime trade-off', loc='left', fontsize=14)
    for label, x, y_val in zip(labels, time_means, acc_means):
        ax_tradeoff.annotate(label.replace('\n', ' '), (x, y_val), xytext=(8, 8), textcoords='offset points', fontsize=9.5, color='#2f2a24')

    style_axis(ax_runs, grid_axis='y')
    box = ax_runs.boxplot(
        acc_runs,
        patch_artist=True,
        labels=labels,
        widths=0.58,
        medianprops={'color': '#1f1b16', 'linewidth': 1.6},
        whiskerprops={'color': '#7b7366'},
        capprops={'color': '#7b7366'}
    )
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.45)
        patch.set_edgecolor(color)
        patch.set_linewidth(1.4)
    for pos, runs, color in zip(np.arange(1, len(method_order) + 1), acc_runs, colors):
        jitter = np.linspace(-0.08, 0.08, len(runs))
        ax_runs.scatter(np.full(len(runs), pos) + jitter, runs, s=42, color=color, edgecolor='#1f1b16', linewidth=0.7, zorder=3)
    ax_runs.set_ylabel('Best OOD accuracy (%)')
    ax_runs.set_title('Run-to-run stability', loc='left', fontsize=14)

    fig.savefig('cmnist_results_dashboard.png', dpi=220, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

# ==========================================
# 4. 执行测试
# ==========================================
num_runs = 5
methods = ['IRMv1', 'Full-BIRM (Official)', 'LoRA-BIRM (Route A)', 'LoRA-BIRM (Route B)']
results = {m: {'acc': [], 'time': []} for m in methods}

print(f"\n⚙️ 启动终极融合测试 (共 {num_runs} 轮, 每轮 2000 步)...")
for run in range(num_runs):
    print(f"\n--- 🔄 第 {run+1}/{num_runs} 轮 ---")
    for method in methods:
        acc, t = run_experiment(method)
        results[method]['acc'].append(acc)
        results[method]['time'].append(t)
        print(f"[{method:22s}] Best OOD Acc: {acc:.2f}% | Time: {t:.2f}s")

print("\n" + "="*70)
print(f"📊 最终实验结果 (5次独立运行) - CMNIST Ultimate Ablation")
print("="*70)
print(f"| {'Method':<22} | {'Test Accuracy (%)':<20} | {'Runtime (s)':<15} |")
print(f"|{'-'*24}|{'-'*22}|{'-'*17}|")
for method in methods:
    acc_str = f"{np.mean(results[method]['acc']):.2f} ± {np.std(results[method]['acc']):.2f}"
    print(f"| {method:<22} | {acc_str:<20} | {np.mean(results[method]['time']):.1f}s |")
print("="*70)

plot_results_dashboard(results)